In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = "rca100/cases/t089"
task_file_path = f"{task_directory}/task.json"

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get("alert_title")
alert_window = task_data.get("alert_window")
alert_entity = task_data.get("alert_entity")

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between 2026-03-27 18:08:44 and 2026-03-27 18:12:01 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}".

The original code execution output of IPython Kernel is also provided below for reference:

('inventory错误次数突增', {'start': '2026-03-27T18:08:44.257676+08:00', 'end': '2026-03-27T18:12:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f"{task_directory}/metrics.parquet"
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs (metric names) and their associated entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates().reset_index(drop=True)
kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs (metric names) across various entity sets. Examples include `node_ready_status` and `node_memory_usage_rate` under `k8s.node`, as well as `error_rate` under `apm.operation` and `apm.instance`. For a complete list, further inspection of the data is needed.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0        node_ready_status            k8s.node
1   node_memory_usage_rate            k8s.node
2    node_not_ready_status            k8s.node
3   node_pod_running_count            k8s.node
4     node_disk_usage_rate            k8s.node
..                     ...                 ...
58                workload  apm.service.legacy
59                   error  apm.service.legacy
60                    slow  apm.service.legacy
61              error_rate       apm.operation
62              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data for 'inventory' service and relevant entity sets
relevant_entity_sets = ['apm.operation', 'apm.service.legacy', 'apm.instance']
inventory_metrics = metrics_df[
    (metrics_df['service'] == 'inventory') & 
    (metrics_df['entity_set'].isin(relevant_entity_sets))
]

# Ensure 'value' column is numeric for calculations
inventory_metrics['value'] = pd.to_numeric(inventory_metrics['value'], errors='coerce')

# Calculate global P95 threshold for each KPI across all components
p95_thresholds = inventory_metrics.groupby('metric')['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

p95_thresholds

```
Out[3]:
```
The filtered metrics data for the 'inventory' service and relevant entity sets (`apm.operation`, `apm.service.legacy`, `apm.instance`) resulted in an empty dataset. This indicates that there are no matching KPIs for the 'inventory' service within these entity sets in the `metrics.parquet` file. Consequently, no global P95 thresholds could be calculated.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [metric, p95_threshold]
Index: []```
```

In [ ]:
# In[4]:

from pytz import timezone
from datetime import datetime

# Load the traces.parquet file
traces_file_path = f"{task_directory}/traces.parquet"
traces_df = pd.read_parquet(traces_file_path)

# Convert alert time window to UTC+8 timestamps in nanoseconds
alert_start = int(datetime.fromisoformat(alert_window['start']).astimezone(timezone('Asia/Shanghai')).timestamp() * 1_000_000_000)
alert_end = int(datetime.fromisoformat(alert_window['end']).astimezone(timezone('Asia/Shanghai')).timestamp() * 1_000_000_000)

# Ensure relevant columns are numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces for 'inventory' service within the alert time window
inventory_traces = traces_df[
    (traces_df['serviceName'] == 'inventory') &
    (traces_df['startTime'] >= alert_start) &
    (traces_df['endTime'] <= alert_end)
]

# Identify traces with errors or high latency
error_traces = inventory_traces[(inventory_traces['statusCode'] != 0) | (inventory_traces['duration'] > inventory_traces['duration'].quantile(0.95))]

error_traces

```
Out[4]:
```
The filtered `traces.parquet` data contains 2,893 traces related to the 'inventory' service during the alert time window. These traces include instances of errors or high latency, indicating potential issues with the `/api/v1/inventory/{productId}` operation. Further analysis of these traces is needed to identify the root cause of the problem.

The original code execution output of IPython Kernel is also provided below for reference:

traceId            spanId      parentSpanId kind                       spanName  ...  __topic__      __tag__:__pack_id__  events           ip links
398081  402d0fec99aa491b3eae374efa48ff5b  e5263d12b90fbed2  85c841850caff61e    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398082  1006a9573a97dcbbe574311ce1895a86  0336e5770db71156  611e7b154e894d2b    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398084  1905413a0ca7fd0ef655a7990f8fe623  2bb810a4815a4a12  73fa80b30ded60dc    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398085  7d88fe6e2df598e57cc0965d8fab6246  f8498f88f7056dca  b789e128ac184ec3    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
398086  d67630bb0697de880fe1a9f97bd9f687  6484f1b45ba4a9b3  e6a2fa4c6f054ffe    2  /api/v1/inventory/{productId}  ...             24387391EE859AA8-215E62    None  10.0.16.164  None
...                                  ...               ...               ...  ...                            ...  ...        ...                      ...     ...          ...   ...
493660  66491a0d24a11ea350e833240921901c  05ec47b4267a9207  dd1eba8ce422e493    2  /api/v1/inventory/{productId}  ...             81C2FCBA3A3F14F2-20D5BB    None  10.0.16.164  None
493661  8d2c980524d598f74aa551aaa3ae6270  56613098ae050f07  3ab1bf02d71d0f58    2  /api/v1/inventory/{productId}  ...             81C2FCBA3A3F14F2-20D5BB    None  10.0.16.164  None
493662  6fc50f8e88294e35dc153557d3b1743f  6686729fcdc0dc4c  dba9e38291bd68f9    2  /api/v1/inventory/{productId}  ...             81C2FCBA3A3F14F2-20D5BB    None  10.0.16.164  None
493663  be605ea3f02ed83e90e1e01120024c25  34dba2747b2d46a1  0bed62a70912efa3    2  /api/v1/inventory/{productId}  ...             81C2FCBA3A3F14F2-20D5BB    None  10.0.16.164  None
498009  ae6951c0382adef4bea452472f193663  23a56c7ad8e9c2cd  bf323b159d33fb4b    2  /api/v1/inventory/{productId}  ...             859609BD1D0CFE0C-20BAFC    None   10.0.2.172  None

[2893 rows x 21 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Extract relevant details for traces with errors or high latency
faulty_traces = error_traces[
    (error_traces['statusCode'] != 0) | 
    (error_traces['statusMessage'].notna()) | 
    (error_traces['duration'] > error_traces['duration'].quantile(0.95))
]

# Select relevant columns for analysis
faulty_traces_details = faulty_traces[['traceId', 'spanName', 'serviceName', 'statusCode', 'statusMessage', 'duration']]

faulty_traces_details

```
Out[5]:
```
The analysis of the filtered traces reveals 2,893 faulty traces related to the 'inventory' service, specifically the `/api/v1/inventory/{productId}` operation. These traces exhibit errors (statusCode = 2) and high latency, with durations ranging up to 15,824,510 nanoseconds. The 'inventory' service appears to be the most downstream faulty service in the call chain during the alert time window. Further investigation into the root cause of these errors and latency is recommended.

The original code execution output of IPython Kernel is also provided below for reference:

traceId                       spanName serviceName statusCode statusMessage  duration
398081  402d0fec99aa491b3eae374efa48ff5b  /api/v1/inventory/{productId}   inventory          2                 1949435
398082  1006a9573a97dcbbe574311ce1895a86  /api/v1/inventory/{productId}   inventory          2                 1582029
398084  1905413a0ca7fd0ef655a7990f8fe623  /api/v1/inventory/{productId}   inventory          2                 1554905
398085  7d88fe6e2df598e57cc0965d8fab6246  /api/v1/inventory/{productId}   inventory          2                 1710893
398086  d67630bb0697de880fe1a9f97bd9f687  /api/v1/inventory/{productId}   inventory          2                 1374372
...                                  ...                            ...         ...        ...           ...       ...
493660  66491a0d24a11ea350e833240921901c  /api/v1/inventory/{productId}   inventory          2                 1195278
493661  8d2c980524d598f74aa551aaa3ae6270  /api/v1/inventory/{productId}   inventory          2                 1256035
493662  6fc50f8e88294e35dc153557d3b1743f  /api/v1/inventory/{productId}   inventory          2                  686946
493663  be605ea3f02ed83e90e1e01120024c25  /api/v1/inventory/{productId}   inventory          2                 1415816
498009  ae6951c0382adef4bea452472f193663  /api/v1/inventory/{productId}   inventory          2                15824510

[2893 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = f"{task_directory}/logs.parquet"
logs_df = pd.read_parquet(logs_file_path)

# Convert alert time window to UTC+8 timestamps in seconds
alert_start_seconds = int(datetime.fromisoformat(alert_window['start']).astimezone(timezone('Asia/Shanghai')).timestamp())
alert_end_seconds = int(datetime.fromisoformat(alert_window['end']).astimezone(timezone('Asia/Shanghai')).timestamp())

# Ensure '_time_' column is numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs for the 'inventory' service during the alert time window
inventory_logs = logs_df[
    (logs_df['_time_'] >= alert_start_seconds) &
    (logs_df['_time_'] <= alert_end_seconds) &
    (logs_df['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Focus on logs containing error messages or anomalies
error_logs = inventory_logs[inventory_logs['content'].str.contains('error|exception|fail|anomaly', case=False, na=False)]

error_logs

```
Out[6]:
```
The filtered logs for the 'inventory' service during the alert time window did not contain any error messages or anomalies. This indicates that no relevant issues were logged in the `logs.parquet` file that could directly explain the observed errors or high latency in the 'inventory' service.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```